# Exact BERT sentence probe and repeat request

Recorded pretrained CPU inference on the exact supplied text, with no audio, target answer or Qwen. The separate live checks submit real audio to the local pipeline without a reference answer or TTS. These examples demonstrate routing, not a semantic-accuracy benchmark.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
while not (ROOT/'scripts/bert_transcript_recovery.py').is_file():
    if ROOT.parent == ROOT: raise RuntimeError('Open this notebook inside the project')
    ROOT = ROOT.parent
OUT = ROOT/'data/generated/bert-sentence-probe-v1'
probe = json.loads((OUT/'result.json').read_text(encoding='utf-8'))
live = json.loads((OUT/'live-check.json').read_text(encoding='utf-8'))
print('BERT forward:', probe['execution']['model_class'], '| CPU | no training')

BERT forward: BertForMaskedLM | CPU | no training


## Actual text result

The output below is the recorded model result, not an authored corrected sentence. The model abstained from choosing a replacement.

In [2]:
r = probe['recovery']
display(pd.DataFrame([{'input': probe['input'], 'BERT output': r['text'], 'status': r['status'], 'reason': r['reason'], 'seconds': r['latency_sec'], 'weights loaded': r['model_loaded']}]))
display(pd.DataFrame(probe['dictionary_membership_separate_diagnostic']))

,input,BERT output,status,reason,seconds,weights loaded
0,I will wi to order a cupf coffee and one ice t...,I will wi to order a cupf coffee and one ice t...,ambiguous,candidate_ambiguity,4.281,True


,index,word,cmudict_entry
0,0,I,True
1,1,will,True
2,2,wi,False
3,3,to,True
4,4,order,True
5,5,a,True
6,6,cupf,False
7,7,coffee,True
8,8,and,True
9,9,one,True


## Candidate decision

Dictionary membership is a lookup result, not proof that ASR recognized the intended word. Masked-LM score margins do not represent an acoustic confidence probability.

In [3]:
for span in r.get('evidence', []):
    print(span['original'], '|', span['decision'], '| selected:', span['selected'])
    display(pd.DataFrame(span['candidates']))
print('Probe source hashes unchanged:', probe['source_unchanged'])

cupf | insufficient_context_gain | selected: None


,text,origin,char_edit_distance,phone_edit_distance,masked_lm_mean_log_score,selection_score
0,cupo,dictionary_spelling,1,None,-6.20242,-6.45242
1,cupf,original,0,None,-6.70804,-6.70804
2,cupp,dictionary_spelling,1,None,-7.34634,-7.59634
3,cups,dictionary_spelling,1,None,-9.76323,-10.01323
4,cuff,dictionary_spelling,1,None,-17.77365,-18.02365


Probe source hashes unchanged: True


## Deployed behavior on real recordings

Unclear recognition uses the application repeat request without a Qwen dialogue attempt. Clear recognition continues through Qwen. Pronunciation and grammar remain in the existing collapse; sentence examples still use the selected reference.

In [4]:
display(pd.DataFrame([{'case': c['name'], 'recognized': c['job']['speech']['transcript'], 'BERT status': c['job']['speech']['interpretation']['status'], 'ask again': c['job']['speech']['transcript_clarity']['needs_clarification'], 'reply': c['job']['text'], 'dialogue attempts': c['job']['metadata']['reply_generation_attempts'], 'grammar when reply appeared': c['first_reply']['rewrite_status']} for c in live['cases']]))
assert live['status'] == 'passed'
assert live['source_sha256'] == live['source_sha256_after']
print('Audio sent externally:', live['audio_sent_externally'], '| TTS calls:', live['voice_api_calls'])

,case,recognized,BERT status,ask again,reply,dialogue attempts,grammar when reply appeared
0,unclear_previous_user_recording,And thoug they lo the for samon play arms,ambiguous,True,"Sorry, I didn't catch that. Could you say it a...",0,pending
1,clear_heldout_reading,A moment before he was intoxicated by joy that...,unchanged,False,That moment was a powerful reminder of the pow...,1,pending


Audio sent externally: False | TTS calls: 0
